# P00 — Preflight de Codespaces y Bedrock

Este notebook comprueba el entorno sin introducir todavía conceptos del curso. Ejecuta las celdas en orden.

In [ ]:
import sys, platform
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

In [ ]:
from pathlib import Path
import sqlite3

start = Path.cwd().resolve()
root = next((p for p in [start, *start.parents] if (p / 'pyproject.toml').exists()), None)
if root is None:
    raise RuntimeError('No se ha encontrado la raíz del repositorio')

db = root / 'data' / 'enterprise.db'
with sqlite3.connect(db) as conn:
    rows = conn.execute('SELECT name, tier, region FROM customers ORDER BY name').fetchall()
rows

## AWS y Bedrock

La siguiente celda comprueba que la API key de Bedrock está disponible como variable de entorno. La clave no se muestra ni se guarda en el notebook.

In [ ]:
import os
import boto3
from enterprise_genai.config import get_settings

settings = get_settings()
token_present = bool(os.getenv('AWS_BEARER_TOKEN_BEDROCK'))
print('Bedrock API key:', 'OK' if token_present else 'MISSING')
print('Region:', settings.aws_region)
print('Text model:', settings.bedrock_text_model_id)

if not token_present:
    raise RuntimeError('AWS_BEARER_TOKEN_BEDROCK no está configurado')

bedrock = boto3.client('bedrock', region_name=settings.aws_region)
models = bedrock.list_foundation_models().get('modelSummaries', [])
print('Foundation models visible:', len(models))

## Primera invocación real

Esta celda realiza una llamada real a Amazon Bedrock mediante la API Converse y muestra también consumo de tokens y latencia.

In [ ]:
from enterprise_genai.models import converse_text

response = converse_text(
    'Resume esta incidencia en una frase: 73 usuarios de ACME no pueden acceder a la VPN desde las 08:15.',
    system_prompt='Eres un asistente de soporte empresarial. Responde de forma breve.',
    temperature=0.2,
    max_tokens=120,
)
print(response['text'])
print('Usage:', response['usage'])
print('Metrics:', response['metrics'])